## 1. Data Loading, Cleaning, and Preparation

In [41]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.decomposition import PCA
import numpy as np

# Load the data
try:
    df = pd.read_csv('./house-prices-advanced-regression-techniques/train.csv')
except FileNotFoundError:
    print("Error: housing_data.csv not found. Make sure the file is in the same directory or provide the correct path.")
    exit()

# Drop the "Id" column
df = df.drop("Id", axis=1)

# Handle missing values
for col in df.columns:
    if df[col].isnull().sum() / len(df) > 0.4:
        df = df.drop(col, axis=1)

for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(exclude=np.number).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# Convert categorical features to dummy variables
df = pd.get_dummies(df, drop_first=True)

# Split into training and testing sets
X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 2. Linear Regression (Original Data)

In [44]:
# Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Linear Regression (Original Data):\nR2: {r2:.4f}\nRMSE: {rmse:.4f}")


Linear Regression (Original Data):
R2: 0.6478
RMSE: 51973.1381


## 3. PCA Transformation

In [47]:
# PCA
pca = PCA(n_components=0.9, random_state=42)
pca.fit(X_train)
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)
num_features_pca = X_train_pca.shape[1]
print(f"Number of features after PCA: {num_features_pca}")


Number of features after PCA: 1


## 4. Linear Regression (PCA Transformed Data)

In [50]:
# Linear Regression (PCA)
model_pca = LinearRegression()
model_pca.fit(X_train_pca, y_train)
y_pred_pca = model_pca.predict(X_test_pca)
r2_pca = r2_score(y_test, y_pred_pca)
rmse_pca = np.sqrt(mean_squared_error(y_test, y_pred_pca))

print(f"Linear Regression (PCA):\nR2: {r2_pca:.4f}\nRMSE: {rmse_pca:.4f}")


Linear Regression (PCA):
R2: 0.0635
RMSE: 84754.5802


## 5. Min-Max Scaling and Feature Selection

In [53]:
# Min-Max Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Variance thresholding
variances = np.var(X_train_scaled, axis=0)
high_variance_indices = variances > 0.1
X_train_high_variance = X_train_scaled[:, high_variance_indices]
X_test_high_variance = X_test_scaled[:, high_variance_indices]

print(f"Number of features with variance > 0.1: {X_train_high_variance.shape[1]}")


Number of features with variance > 0.1: 227


In [55]:
# Separate features (X) and target (y)
X = df[['X']]
y = df['y']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create polynomial features (high degree to induce high variance/overfitting)
degree = 10  # Experiment with different degrees
poly = PolynomialFeatures(degree=degree, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Create and train a Linear Regression model on the polynomial features
poly_reg_model = LinearRegression()
poly_reg_model.fit(X_train_poly, y_train)

# Make predictions
y_train_pred = poly_reg_model.predict(X_train_poly)
y_test_pred = poly_reg_model.predict(X_test_poly)

# Evaluate the model
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"Polynomial Regression (Degree {degree}) - Train RMSE: {train_rmse:.2f}")
print(f"Polynomial Regression (Degree {degree}) - Test RMSE: {test_rmse:.2f}")

# Plotting the results (optional)
plt.scatter(X_train, y_train, label='Training Data')
plt.scatter(X_test, y_test, label='Test Data')
plt.plot(X_train, y_train_pred, color='red', label='Polynomial Regression Fit (Training)')
# You might want to plot the test predictions as well, but it might not be very insightful
# given the potential overfitting.
plt.title(f'Polynomial Regression (Degree {degree})')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()

KeyError: "None of [Index(['X'], dtype='object')] are in the [columns]"